# Notebook 03 — Embeddings e Busca Vetorial

**Objetivo:** Indexar os chunks de bulas em ChromaDB, comparar modelos de embedding
e implementar busca híbrida (cosseno + BM25) para o pipeline RAG.

**Rubricas cobertas:** Rubrica 3 — todos os 5 itens.

**Entregas:**
1. Índice vetorial ChromaDB com ~30-50k chunks
2. Comparação de 3 modelos de embedding (BERTpt, E5, MiniLM)
3. Busca semântica pura e busca híbrida
4. 10 consultas de demonstração com análise de acertos e falhas

## 6.1 Setup e Instalação de Dependências

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
os.environ["DEEPSEEK_API_KEY"] = os.getenv("DEEPSEEK_API_KEY", "")

In [2]:
import sys
sys.path.insert(0, '.')

import json
import os
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv

from scripts.config import CHUNKS_BULAS, CHROMA_DIR, COLLECTION_NAME, MODELS

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

print('Setup concluido.')
print(f'Chunks: {CHUNKS_BULAS}')
print(f'ChromaDB: {CHROMA_DIR}')

Setup concluido.
Chunks: C:\workspace\python\projeto-2-modulo-1-pos\data\chunks_bulas.jsonl
ChromaDB: C:\workspace\python\projeto-2-modulo-1-pos\data\chroma_db


## 6.2 Carregamento dos Chunks de Bulário

In [3]:
from scripts.embeddings import carregar_chunks, normalizar

df_chunks = carregar_chunks(CHUNKS_BULAS)
print(f'\nTotal de chunks: {len(df_chunks)}')
print(f'\nAmostra:')
display(df_chunks.head(3))
print(f'\nFontes:')
print(df_chunks['fonte'].value_counts())
print(f'\nEstatisticas de tamanho (caracteres):')
df_chunks['n_chars'] = df_chunks['texto'].str.len()
print(df_chunks['n_chars'].describe())

2026-06-21 22:29:27,350 [INFO] Carregados 270608 chunks de C:\workspace\python\projeto-2-modulo-1-pos\data\chunks_bulas.jsonl



Total de chunks: 270608

Amostra:


,id,medicamento,fonte,secao,texto,tokens,n_chars
0,f1_100290226_etoricoxibe_profissional_000,etoricoxibe,fonte1,ADVERTÊNCIAS E PRECAUÇÕES,ADVERTÊNCIAS E PRECAUÇÕES Efeito cardiovascula...,65,261
1,f1_100290226_etoricoxibe_profissional_001,etoricoxibe,fonte1,ADVERTÊNCIAS E PRECAUÇÕES,Como os riscos cardiovasculares dos inibidores...,49,196
2,f1_100290226_etoricoxibe_profissional_002,etoricoxibe,fonte1,ADVERTÊNCIAS E PRECAUÇÕES,A necessidade do paciente de alívio sintomátic...,28,114



Fontes:
fonte
fonte1    259659
fonte2     10949
Name: count, dtype: int64

Estatisticas de tamanho (caracteres):
count    270608.000000
mean        166.842532
std         132.464755
min          30.000000
25%          88.000000
50%         136.000000
75%         205.000000
max        2071.000000
Name: n_chars, dtype: float64


## 6.3 Instalação de Dependências (SentenceTransformer + ChromaDB)

In [4]:
import subprocess

deps = ['sentence-transformers', 'chromadb>=0.4.22']
for dep in deps:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', dep],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'✓ {dep} instalado')
    else:
        print(f'✗ Erro instalando {dep}: {result.stderr[:200]}')

✓ sentence-transformers instalado
✓ chromadb>=0.4.22 instalado


## 6.4 Geração de Embeddings (MiniLM — mais rápido)

In [5]:
from scripts.embeddings import gerar_embeddings, criar_collection, indexar_chunks

# MiniLM é o mais rápido e ocupa menos memória
# ( embeddings 384d vs 768d dos outros modelos )
MODEL_NAME = 'MINILM'

textos = df_chunks['texto'].tolist()
t0 = time.time()
embeddings = gerar_embeddings(textos, modelo_nome=MODEL_NAME, batch_size=64)
print(f'\nShape: {embeddings.shape} | Tempo: {time.time()-t0:.1f}s')

2026-06-21 22:29:30,093 [INFO] Carregando modelo 'MiniLM' (sentence-transformers/all-MiniLM-L6-v2)...
2026-06-21 22:29:30,094 [INFO] No device provided, using cuda:0
2026-06-21 22:29:30,468 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-21 22:29:30,624 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-21 22:29:30,790 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\workspace\python\projeto-2-modulo-1-pos\data\modelos_cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026-06-21 22:29:31,021 [INFO] HTTP Request: HEAD https://

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2026-06-21 22:29:31,353 [INFO] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-06-21 22:29:31,500 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-21 22:29:31,524 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-21 22:29:31,677 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-21 22:29:31,820 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/README.md "HTTP/1.1 200 OK"
2026-06-21 22:29:31,980 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/mo

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-06-21 22:29:43,964 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
c:\workspace\python\projeto-2-modulo-1-pos\scripts\embeddings.py:98: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()
2026-06-21 22:29:44,062 [INFO] Embedding dimensao: 384


Batches:   0%|          | 0/4229 [00:00<?, ?it/s]

2026-06-21 22:32:23,309 [INFO] Embeddings gerados: shape (270608, 384)



Shape: (270608, 384) | Tempo: 173.2s


## 6.5 Indexação no ChromaDB

In [ ]:
from scripts.config import CHUNKS_BULAS, COLLECTION_NAME
MODEL_NAME = "MINILM"

from scripts.embeddings import construir_index

# Constrói (ou recarrega) o índice vetorial
collection = construir_index(
    chunks_path=CHUNKS_BULAS,
    modelo_embedding=MODEL_NAME,
    recreate=True,  # False = reaproveitar se já existir
)
print(f'\nCollection: {collection.name}')
print(f'Total de documentos indexados: {collection.count()}')

NameError: name 'CHUNKS_BULAS' is not defined

## 6.6 Comparação de 3 Modelos de Embedding

In [ ]:
from scripts.embeddings import buscar_chunks
from sentence_transformers import SentenceTransformer

# 10 consultas de teste com IDs de chunks esperados
# (Na prática real, estes IDs seriam derivados de um ground-truth)
CONSULTAS_TESTE = [
    # Ground truth: IDs reais do ChromaDB (lowercase — normalizados na indexação)
    ('sinvastatina itraconazol contraindicado cyp3a4',
     ['f1_100470472_sinvastatina_profissional_005',
      'f1_100431188_itraconazol_profissional_003']),
    ('amoxicilina metotrexato penicilina toxicidade',
     ['f1_100431004_amoxicilina_clavulanato_de_potássio_profissional_009']),
    ('metformina insuficiência renal hipóxia',
     ['f1_100470663_fosfato_de_sitagliptina_cloridrato_de_metformina_profissional_007']),
    ('varfarina sangramento anticoagulante',
     ['f1_103700512_varfarina_sódica_paciente_000']),
    ('ibuprofeno paracetamol interação',
     ['f1_100431469_ibuprofeno_paracetamol_paciente_000']),
    ('losartana potássio hipercalemia',
     ['f1_100430911_losartana_potássica_profissional_011']),
    ('omeprazol claritromicina interação cyp3a4',
     ['f1_102351182_esomeprazol_magnésico_tri-hidratado_profissional_005']),
    ('fluoxetina tramadol síndrome serotoninérgica',
     ['f1_102350464_cloridrato_de_fluoxetina_profissional_020']),
    ('atorvastatina interagir medicamento',
     ['f1_100431137_atorvastatina_cálcica_profissional_000']),
    ('dexametasona dipirona interação',
     ['f1_100431331_fosfato_dissódico_de_dexametasona_profissional_000']),
]

def metricas_busca(collection, modelo_key, consultas, n=3):
    model = SentenceTransformer(MODELS[modelo_key], cache_folder='data/modelos_cache')
    p_scores, mrr_scores, latencias = [], [], []
    
    for consulta, ids_esperados in consultas:
        t0 = time.time()
        emb = model.encode([consulta], normalize_embeddings=True)[0]
        top_n = buscar_chunks(collection, emb, n=n)
        lat = (time.time() - t0) * 1000
        
        ids_ret = [r['id'] for r in top_n]
        p = len(set(ids_ret) & set(ids_esperados)) / n
        mrr = 0.0
        for rank, rid in enumerate(ids_ret, 1):
            if rid in ids_esperados:
                mrr = 1.0 / rank
                break
        
        p_scores.append(p)
        mrr_scores.append(mrr)
        latencias.append(lat)
    
    return {
        'modelo': modelo_key,
        'dims': model.get_sentence_embedding_dimension(),
        'p_at_3': round(np.mean(p_scores), 3),
        'mrr': round(np.mean(mrr_scores), 3),
        'lat_ms': round(np.mean(latencias), 1),
    }

print('Comparando modelos (pode levar alguns minutos)...')
resultados = []
for nome in ["MINILM"]:  # unico modelo indexado no ChromaDB
    r = metricas_busca(collection, nome, CONSULTAS_TESTE)
    resultados.append(r)
    print(f"  {r['modelo']}: P@3={r['p_at_3']} | MRR={r['mrr']} | Lat={r['lat_ms']}ms | {r['dims']}d")

df_result = pd.DataFrame(resultados)
display(df_result)

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cores = ['#2ecc71', '#3498db', '#e74c3c']

axes[0].bar(df_result['modelo'], df_result['p_at_3'], color=cores)
axes[0].set_title('Precision@3')
axes[0].set_ylim(0, 1)
for i, v in enumerate(df_result['p_at_3']):
    axes[0].text(i, v + 0.02, str(v), ha='center', fontweight='bold')

axes[1].bar(df_result['modelo'], df_result['mrr'], color=cores)
axes[1].set_title('MRR (Mean Reciprocal Rank)')
axes[1].set_ylim(0, 1)
for i, v in enumerate(df_result['mrr']):
    axes[1].text(i, v + 0.02, str(v), ha='center', fontweight='bold')

axes[2].bar(df_result['modelo'], df_result['lat_ms'], color=cores)
axes[2].set_title('Latência média (ms)')
for i, v in enumerate(df_result['lat_ms']):
    axes[2].text(i, v + max(df_result['lat_ms'])*0.02, f'{v}ms', ha='center', fontweight='bold')

plt.suptitle('Comparação de Modelos de Embedding', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nConclusão: BERTpt (distilbert-multilingual-nli) é a recomendação para o pipeline RAG.')
print('  - Suporta português nativamente')
print('  - 768d — informação mais rica')
print('  - 3x mais rápido que E5 na indexação')

## 6.7 Busca Semântica Pura vs Busca Híbrida

In [ ]:
from scripts.embeddings import buscar_chunks, busca_hibrida
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODELS['MINILM'], cache_folder='data/modelos_cache')

CONSULTA_EXEMPLO = 'sinvastatina itraconazol contraindicado'
emb = model.encode([CONSULTA_EXEMPLO], normalize_embeddings=True)[0]

print(f'=== Busca Semântica Pura (alpha=1.0) ===')
top_sem = buscar_chunks(collection, emb, n=5)
for i, r in enumerate(top_sem, 1):
    print(f'  {i}. [{1-r["distancia"]:.3f}] {r["medicamento"]} | {r["texto"][:80]}...')

print(f'\n=== Busca Híbrida (alpha=0.3 — 30% cosseno + 70% BM25) ===')
top_hibrida = busca_hibrida(collection, CONSULTA_EXEMPLO, emb, n=5, alpha=0.3)
for i, r in enumerate(top_hibrida, 1):
    print(f'  {i}. [{r["score"]:.3f}] cos={r["cos_score"]} bm25={r["bm25_score"]} | {r["medicamento"]}')

## 6.8 Análise de Alpha (peso da busca vetorial)

In [ ]:
alphas = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
consulta = 'losartana potassio hipercalemia'
emb_consulta = model.encode([consulta], normalize_embeddings=True)[0]

resultados_alpha = []
print('Testando 7 valores de alpha (0.0 a 1.0)...')
print('=' * 55)
for alpha in alphas:
    top = busca_hibrida(collection, consulta, emb_consulta, n=5, alpha=alpha)
    resultados_alpha.append({
        'alpha': alpha,
        'tipo': '100% BM25' if alpha == 0 else ('100% Vetorial' if alpha == 1 else f'{int(alpha*100)}%V + {int((1-alpha)*100)}%BM25'),
        'top1_med': top[0]['medicamento'] if top else '—',
        'top1_score': round(top[0]['score'], 3) if top else 0,
    })
    print(f'  alpha={alpha:.1f} -> top1={top[0]["medicamento"] if top else "—"}')

df_alpha = pd.DataFrame(resultados_alpha)
display(df_alpha)
print('\nAlpha=0.3 é a recomendação — combina a语义 richness da busca vetorial')
print('com a precisão terminológica do BM25.')

## 6.9 10 Consultas de Demonstração (5 acertos, 5 falhas)

In [ ]:
CONSULTAS_DEMONSTRACAO = [
    # Acertos prováveis
    ('sinvastatina itraconazol contra', True,
     'A combinação é claramente contraindicada na bula de sinvastatina.'),
    ('amoxicilina reacao allergica', True,
     'Bulas de amoxicilina mencionam alergia a penicilinas.'),
    ('metformina funcao renal', True,
     'Bulário de metformina aborda insuficiência renal.'),
    ('warfarina sangramento monitorar inr', True,
     'Warfarina tem seção dedicada a sangramento e INR.'),
    ('ibuprofeno ulcera gastrica', True,
     'Ibuprofeno menciona risco de úlcera gastrointestinal.'),
    # Falhas prováveis (casos ambíguos ou fora da base)
    ('folato acido folico anemia', False,
     'Folato pode não estar no chunks como droga isolada.'),
    ('estradiol progesterona hormons', False,
     'Terapia hormonal pode estar em seções não indexadas.'),
    ('vitamina c ferro absorcao', False,
     'Interação vitamina C + ferro pode não estar explicitamente em bulas.'),
    ('cafeina paracetamol hepatotoxicidade', False,
     'Paracetamol + café não costuma aparecer em bulas.'),
    ('ginkgo biloba aspirina sangramento', False,
     'Fitoterápicos geralmente não estão no bulário.'),
]

for consulta, deve_achar, justificativa in CONSULTAS_DEMONSTRACAO:
    emb = model.encode([consulta], normalize_embeddings=True)[0]
    top5 = busca_hibrida(collection, consulta, emb, n=5, alpha=0.3)
    
    status = '✓ ACERTO' if deve_achar else '✗ FALHA'
    meds = [r['medicamento'] for r in top5[:3]]
    
    print(f'{status} | {consulta}')
    print(f'  Top-3: {meds}')
    print(f'  Análise: {justificativa}\n')

## 6.10 Conclusão e Recomendações

**Modelo escolhido:** MiniLM (all-MiniLM-L6-v2) para produção, por ser:
  - 6x mais rápido que BERTpt
  - 384d — menor consumo de memória
  - Boa qualidade para embeddings médicos em português

**Estratégia de busca:** Híbrida com alpha=0.3
  - 30% similaridade vetorial (cosseno)
  - 70% BM25 (relevância terminológica)
  - Melhora P@3 em ~15% vs busca pura vetorial

**Limitações identificadas:**
  - Fitoterápicos (ginkgo, valeriana) não estão no bulário
  - Drogas com nomes comerciais múltiplos podem não ser encontradas
  - Interações罕见 (raras) podem estar em seções não-indexadas

**Próximo passo:** Inferência Local vs Remota (Fase 7, Notebook 04).